# BoxDistractor (`Box-aside-v0`) — train paper-faithful checkpoint

**Long-horizon manipulation** (move a distractor box aside, then pick/place) — the
manipulation analogue of AntMaze's long-horizon navigation. Complements the short-horizon
`FetchPickAndPlace` so the ablation covers *both* a hard-nav and a hard-manip env.

Entry point is **`rl.main_latent_robot`** (NOT `main_latent_fetch`) — launcher
`launcher_latent_robot.py` (`Agent=latent_fetch`, `Algo=latent_planner`, `import fetch`).
Flags copied verbatim from the paper's `scripts/distractor.sh` (seed 829; paper also
runs 252/173). Checkpoints land in `.../Box-aside-v0/boxdistractor_s829/state/`.

**Before running:** Settings → Accelerator = **GPU T4**, Internet = **ON**.
To *continue* a prior run: **Add Data →** the output notebook that saved `boxdistractor_s829` (weights
+ optimizer + replay), then set `SLUG` in cell 2. Fresh run: skip cell 2.

## 1. Code + MuJoCo env (~10–15 min first time)

In [ ]:
import os
if os.path.isdir('/kaggle/working/latent_landmarks'):
    !cd /kaggle/working/latent_landmarks && git pull -q origin retrain
else:
    !git clone -q -b retrain https://github.com/Jun1801/latent_landmarks.git /kaggle/working/latent_landmarks
if not os.path.isdir('/kaggle/working/wmag'):
    !git clone -q https://github.com/LunjunZhang/world-model-as-a-graph /kaggle/working/wmag
!bash /kaggle/working/latent_landmarks/repro/setup_kaggle.sh

## 2. (Resume only) Restore prior checkpoint — weights + optimizer + **replay**
Skip entirely for a fresh run. Unlike *eval*, resuming training needs the full state
incl. `replay_0.pt` (~hundreds of MB). Find `<SLUG>` with `!ls /kaggle/input/`.

In [ ]:
!ls /kaggle/input/
import os, shutil
SLUG = 'PUT-DATASET-SLUG-HERE'          # <-- output notebook that saved boxdistractor_s829
CKPT, ENV = 'boxdistractor_s829', 'Box-aside-v0'
src = f'/kaggle/input/{SLUG}/experiments/{ENV}/{CKPT}/state'
dst = f'/kaggle/working/experiments/{ENV}/{CKPT}/state'; os.makedirs(dst, exist_ok=True)
if os.path.isdir(src):
    for f in os.listdir(src):
        shutil.copy(f'{src}/{f}', f'{dst}/{f}')
    print('restored ->', sorted(os.listdir(dst)))
else:
    print('no prior state at', src, '-> fresh run')

## 3. Verify GPU + MuJoCo + env registration

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-}; cd /kaggle/working/wmag && conda run -n l3p python -c "import torch, mujoco_py, gym, fetch; print('cuda', torch.cuda.is_available()); e=gym.make('Box-aside-v0'); print('env ok', e.observation_space['observation'].shape, e.action_space.shape)"

## 4. Train — resume-aware
`--n_epochs` default is 10000 (an upper bound, not a target); capped to **500** here — a
distractor task is harder than plain pick, so watch `Test_TestEnv_PlanSuccessRate` and
**stop when it plateaus**. Re-running this cell auto-adds `--resume_ckpt` if a checkpoint
exists (restores weights/optimizer/replay; the epoch counter restarts at 0 but training
continues). `--n_workers 3` fits Kaggle's ~4 CPUs (paper leaves the default 12).

In [ ]:
%%bash
export PATH=/opt/conda/bin:$PATH
export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-}
cd /kaggle/working/wmag
SAVE=/kaggle/working/experiments; CKPT=boxdistractor_s829
RESUME=""
if [ -f "$SAVE/Box-aside-v0/$CKPT/state/algo.pt" ]; then
  RESUME="--resume_ckpt $CKPT"; echo "[resume] continuing $CKPT"
else
  echo "[fresh] new run $CKPT"
fi
conda run -n l3p python -m rl.main_latent_robot \
  --env_name Box-aside-v0 --test_env_name Box-aside-v0 --cuda \
  --seed 829 --n_workers 3 --n_cycles 15 --clip_inputs --normalize_inputs --gamma 0.99 \
  --n_initial_rollouts 100 --n_test_rollouts 10 --plan_eps 0.5 \
  --n_latent_landmarks 80 --latent_batch_size 150 --n_extra_landmark 20 \
  --dist_clip -15.0 --start_planning_n_traj 6000 --use_forward_empty_step \
  --n_epochs 500 --save_dir $SAVE --ckpt_name $CKPT $RESUME

## 5. Persist for resume / eval
**Save Version** (commit) so `/kaggle/working/experiments/Box-aside-v0/boxdistractor_s829/state/` is written to
the notebook *output*. Next session: Add Data → this output, set `SLUG` in cell 2.
- Resume training: needs `agent.pt` + `algo.pt` + `replay_0.pt`.
- Ablation (eval only): needs just `agent.pt` + `algo.pt` (see `boxdistractor_ablation.ipynb`, later).

In [ ]:
!ls -la /kaggle/working/experiments/Box-aside-v0/boxdistractor_s829/state/ 2>/dev/null || echo 'no checkpoint yet'